#### Initialize

In [8]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
MAP_PATH = PARENT / "server/map/grid"
MAP_PATH.mkdir(parents=True, exist_ok=True)
RANKED_BUCKET_PATH = PARENT / "server/out/places_ranked"
DF_RANKED = pd.read_csv(RANKED_BUCKET_PATH / "places_scored_level_1.csv")

#### Representation

In [9]:
df_ranked = DF_RANKED.copy()
# df_ranked = df_ranked[df_ranked['normal_1']>=0.75]

counts = (
    df_ranked.assign(**{'cuisineType': df_ranked['cuisineType'].fillna('Unspecified')})
      .groupby('cuisineType', dropna=False)
      .size().reset_index(name='row_count')
      .sort_values('row_count', ascending=False)
)

# print(f"Using column: {'cuisineType'}")
# counts.sort_values('row_count', ascending=False)

#### Tier Diversified

In [10]:
import numpy as np
from server.scripts.rank_places_level_2.assign_tier import assign_tier
RANK_IDXS = [0, 1, 2]
# Base tier from normalized Wilson score
for rank_idx in RANK_IDXS:
    df_ranked[f'tier_{rank_idx}'] = assign_tier(df_ranked[f'normal_{rank_idx}'])

    # Diversified tier (tier_d): start from base tier, then clip over-represented cuisines
    df_ranked[f'tier_d{rank_idx}'] = df_ranked[f'tier_{rank_idx}']
    df_ranked['_cuisine_key'] = df_ranked['cuisineType'].fillna('Unspecified')

    score_col = f'normal_{rank_idx}'
    cap_multiplier = 1.2
    min_cuisine_size = 50

    # Global cuisine prevalence
    global_counts = df_ranked['_cuisine_key'].value_counts()
    global_share = (global_counts / len(df_ranked)).to_dict()
    eligible_cuisines = set(global_counts[global_counts > min_cuisine_size].index)

    # Enforce cap per tier from top to bottom.
    # If a cuisine exceeds cap in a tier, demote lowest-scored excess rows to next lower tier.
    for t in sorted(df_ranked[f'tier_d{rank_idx}'].unique(), reverse=True):
        if t == 0:
            continue

        tier_mask = df_ranked[f'tier_d{rank_idx}'] == t
        tier_size = int(tier_mask.sum())
        if tier_size == 0:
            continue

        tier_counts = df_ranked.loc[tier_mask, '_cuisine_key'].value_counts()

        for cuisine, cnt in tier_counts.items():
            if cuisine not in eligible_cuisines:
                continue

            cap = int(np.floor(cap_multiplier * global_share[cuisine] * tier_size))
            cap = max(cap, 1)
            excess = int(cnt - cap)

            if excess <= 0:
                continue

            drop_idx = (
                df_ranked.loc[tier_mask & (df_ranked['_cuisine_key'] == cuisine)]
                .sort_values(score_col, ascending=True)
                .head(excess)
                .index
            )
            df_ranked.loc[drop_idx, f'tier_d{rank_idx}'] = t - 1

    # Cleanup helper column
    df_ranked.drop(columns=['_cuisine_key'], inplace=True)

# Quick check
# display(df_ranked[['tier_0', 'tier_d0']].value_counts().rename('rows').reset_index().sort_values(['tier_0', 'tier_d0']))
# df_ranked[['tier_0', 'tier_d0']].head()

#### Tier NoChain

In [11]:
# tier_i{rank_idx}: recount tiers after removing chain restaurants
chain_mask = df_ranked['is_major_chain'].eq(True)
non_chain_mask = ~chain_mask

for rank_idx in RANK_IDXS:
    tier_column = f'tier_i{rank_idx}'
    score_col = f'normal_{rank_idx}'
    df_ranked[tier_column] = 0
    df_ranked.loc[non_chain_mask, tier_column] = assign_tier(
        df_ranked.loc[non_chain_mask, score_col]
    )

    non_chain_places = df_ranked.loc[non_chain_mask].copy()
    non_chain_places['_cuisine_key'] = non_chain_places['cuisineType'].fillna('Unspecified')
    global_counts = non_chain_places['_cuisine_key'].value_counts()
    global_share = (global_counts / len(non_chain_places)).to_dict()
    eligible_cuisines = set(global_counts[global_counts > min_cuisine_size].index)

    for tier in sorted(non_chain_places[tier_column].unique(), reverse=True):
        if tier == 0:
            continue

        tier_mask = non_chain_places[tier_column].eq(tier)
        tier_size = int(tier_mask.sum())
        if tier_size == 0:
            continue

        tier_counts = non_chain_places.loc[tier_mask, '_cuisine_key'].value_counts()
        for cuisine, count in tier_counts.items():
            if cuisine not in eligible_cuisines:
                continue

            cap = max(int(np.floor(cap_multiplier * global_share[cuisine] * tier_size)), 1)
            excess = int(count - cap)
            if excess <= 0:
                continue

            drop_idx = (
                non_chain_places.loc[
                    tier_mask & non_chain_places['_cuisine_key'].eq(cuisine)
                ]
                .sort_values(score_col, ascending=True)
                .head(excess)
                .index
            )
            non_chain_places.loc[drop_idx, tier_column] = tier - 1

    df_ranked.loc[non_chain_places.index, tier_column] = non_chain_places[tier_column]

In [12]:
# Check the result
display(df_ranked[['tier_0', 'tier_d0', 'tier_i0']].value_counts().rename('rows').reset_index().sort_values(['tier_0', 'tier_d0', 'tier_i0']))
chain_demoted_count = ((df_ranked['is_chain'] == True) & (df_ranked['tier_d0'] > 0)).sum()
print(f"\nChain restaurants demoted to tier 0: {chain_demoted_count}")
df_ranked[['displayName', 'tier_0', 'tier_d0', 'tier_i0', 'is_chain']].head()

,tier_0,tier_d0,tier_i0,rows
0,0,0,0,9119
5,1,0,0,387
9,1,0,1,118
6,1,1,0,264
1,1,1,1,3791
21,2,1,0,3
7,2,1,1,150
12,2,1,2,61
10,2,2,0,111
15,2,2,1,40



Chain restaurants demoted to tier 0: 1626


,displayName,tier_0,tier_d0,tier_i0,is_chain
0,De Vine,4,4,4,False
1,Tofu Vegan Charlotte Street,4,4,4,False
2,Ethical Bean Company Coffee Shop,4,4,4,False
3,Bench Bistro,4,4,4,False
4,Falafel Zaki Zaki,4,4,4,True


#### Export

In [13]:
df_ranked.to_csv(RANKED_BUCKET_PATH / "places_ranked_level_2.csv", index=False)